# Intermediate: NumPy and pandas Patterns Used Throughout AI Work

**Sub-topic 1.3 — Python Refresher for AI**

**Part 1 (NumPy)** previews a concept sub-topic 1.4 covers in full: text "meaning" can be represented as a vector of numbers (an embedding), and comparing meaning becomes comparing vectors mathematically. We use tiny hand-made 3-number vectors here, not real embeddings, purely to build comfort with vector operations before 1.4 makes it rigorous.

**Part 2 (pandas)** covers filtering, grouping, and joining — the operations you'll run on every dataset you touch in this curriculum.

In [1]:
import numpy as np
import pandas as pd

## Part 1 — NumPy: vectors as a stand-in for "meaning"

Pretend these are toy 3-dimensional "embeddings" for three short phrases, where each dimension loosely represents some latent concept (this is **not** how real embeddings are built — it's a hand-picked illustration so the numbers are easy to reason about).

In [2]:
embeddings = {
    "dog":  np.array([0.9, 0.1, 0.0]),
    "puppy": np.array([0.85, 0.15, 0.05]),
    "car":  np.array([0.0, 0.1, 0.95]),
}

for word, vec in embeddings.items():
    print(f"  {word!r:8s} -> {vec}")

  'dog'    -> [0.9 0.1 0. ]
  'puppy'  -> [0.85 0.15 0.05]
  'car'    -> [0.   0.1  0.95]


In [3]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Cosine similarity (1.0 = identical direction, 0.0 = unrelated):")
print(f"  dog vs puppy: {cosine_similarity(embeddings['dog'], embeddings['puppy']):.3f}")
print(f"  dog vs car:   {cosine_similarity(embeddings['dog'], embeddings['car']):.3f}")

Cosine similarity (1.0 = identical direction, 0.0 = unrelated):
  dog vs puppy: 0.996
  dog vs car:   0.012


`dog` and `puppy` point in a similar direction (high similarity); `dog` and `car` don't. This is the exact math vector databases use to find "similar" documents in Track 3 — just with 1000+ dimensional vectors produced by a real embedding model instead of 3 hand-picked ones.

In [4]:
# Vectorized operations -- no Python for-loop needed.
stacked = np.stack(list(embeddings.values()))
print(f"All embeddings stacked into a matrix, shape={stacked.shape}")
print(f"Column-wise mean (average value per dimension): {stacked.mean(axis=0)}")

All embeddings stacked into a matrix, shape=(3, 3)
Column-wise mean (average value per dimension): [0.58333333 0.11666667 0.33333333]


## Part 2 — pandas: filter, group, join

The bread and butter of working with any tabular dataset (which is most of what you'll do outside of raw text).

In [5]:
docs = pd.DataFrame({
    "doc_id": [1, 2, 3, 4, 5],
    "title": ["Intro to AI", "Deep Learning Basics", "Prompt Engineering 101",
               "RAG Systems", "Vector Databases"],
    "category": ["foundations", "foundations", "prompting", "applications", "applications"],
    "word_count": [1200, 2100, 900, 1800, 1500],
})

ratings = pd.DataFrame({
    "doc_id": [1, 2, 3, 4, 5],
    "avg_rating": [4.2, 4.7, 4.9, 4.5, 4.1],
})

docs

,doc_id,title,category,word_count
0,1,Intro to AI,foundations,1200
1,2,Deep Learning Basics,foundations,2100
2,3,Prompt Engineering 101,prompting,900
3,4,RAG Systems,applications,1800
4,5,Vector Databases,applications,1500


### Filter: documents with word_count > 1300

In [6]:
docs[docs["word_count"] > 1300]

,doc_id,title,category,word_count
1,2,Deep Learning Basics,foundations,2100
3,4,RAG Systems,applications,1800
4,5,Vector Databases,applications,1500


### Group by category: average word count

In [7]:
docs.groupby("category")["word_count"].mean()

category
applications    1650.0
foundations     1650.0
prompting        900.0
Name: word_count, dtype: float64

### Join documents with their ratings (on `doc_id`)

In [8]:
merged = docs.merge(ratings, on="doc_id")
merged

,doc_id,title,category,word_count,avg_rating
0,1,Intro to AI,foundations,1200,4.2
1,2,Deep Learning Basics,foundations,2100,4.7
2,3,Prompt Engineering 101,prompting,900,4.9
3,4,RAG Systems,applications,1800,4.5
4,5,Vector Databases,applications,1500,4.1


### Vectorized column math (no for-loop): a "quality score" column

In [9]:
merged["quality_score"] = (merged["avg_rating"] / 5) * 100
merged[["title", "avg_rating", "quality_score"]]

,title,avg_rating,quality_score
0,Intro to AI,4.2,84.0
1,Deep Learning Basics,4.7,94.0
2,Prompt Engineering 101,4.9,98.0
3,RAG Systems,4.5,90.0
4,Vector Databases,4.1,82.0


### Try it yourself
Add a new column that's the ratio of two existing numeric columns, using vectorized pandas operations (no `for` loop).